In [1]:
import os, pickle

In [2]:
with open("../../../training_data/8.Apos/Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pickle.load(f)

len(extras_featuresd), extras_featuresd

(24,
 {'8sgj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       8sgj               1             A           52            A   
  1       8sgj               1             A           53            A   
  2       8sgj               1             A           54            A   
  3       8sgj               1             A           55            A   
  4       8sgj               1             A           56            A   
  ..       ...             ...           ...          ...          ...   
  746     8sgj               1             A          941            A   
  747     8sgj               1             A          942            A   
  748     8sgj               1             A          943            A   
  749     8sgj               1             A          944            A   
  750     8sgj               1             A          945            A   
  
                      

# Make predictions

## Gather data/features

In [3]:
if not os.path.isfile("fpocket"):
    os.system(f"ln -s /home/fnerin/miniconda3/envs/allopockets/bin/fpocket fpocket") # os.getcwd().rsplit('/', 3)[0] --> absolute path of allodb_new

In [4]:
if not os.path.isfile("mkdssp-4.4.0-linux-x64"):
    os.system(f"ln -s {os.getcwd().rsplit('/', 3)[0]}/training_data/utils/external/mkdssp-4.4.0-linux-x64 mkdssp-4.4.0-linux-x64") # os.getcwd().rsplit('/', 3)[0] --> absolute path of allodb_new

Ran with run_psiblast.py

In [5]:
import subprocess, pymol2

In [6]:
for pdb in sorted(extras_featuresd):
    # if pdb == "7sns": continue
    pdb = pdb.upper()
    os.makedirs(pdb, exist_ok=True)

    # os.system(f"ln -s {os.getcwd().rsplit('/', 1)[0]}/structures/{pdb.lower()}.pdb {pdb}/{pdb}.pdb") # pdb file needs a HEADER first line for mkdssp to work
    pdbf = f"{pdb}/{pdb}.pdb"
    if not os.path.isfile(pdbf):
        with (
            open(f"{os.getcwd().rsplit('/', 1)[0]}/structures/{pdb.lower()}.pdb", "r") as orig_pdbf,
            open(pdbf, "w") as f
        ):
            f.write(f"HEADER {pdb}\n")
            f.write(orig_pdbf.read())
    
    # os.system(f"./mkdssp-4.4.0-linux-x64 --output-format dssp {pdb}/{pdb}.pdb") # need to capture output
    dsspf = pdbf.replace(".pdb", ".dssp")
    if not os.path.isfile(dsspf):
        with open(dsspf, "w") as f:
            f.write(
                subprocess.run(["./mkdssp-4.4.0-linux-x64", "--output-format=dssp", pdbf], capture_output=True)
                .stdout.decode()
            )

    asnf = pdbf.replace(".pdb", ".asn")
    if not os.path.isfile(asnf):
        for f in os.listdir(pdb):
            if f.endswith("-PSSM_Scoremat.asn"):
                os.system(f"ln -s {f} {asnf}")
                
    if not os.path.isfile(asnf):
        print(asnf)
        with pymol2.PyMOL() as pymol, open(pdbf.replace(".pdb", ".fasta"), "w") as f:#tempfile.NamedTemporaryFile("w+", suffix=".fasta") as f:
            pymol.cmd.load(pdbf, "prot")
            f.write(pymol.cmd.get_fastastr('prot'))
        
        # os.system(f"cd nr && psiblast -query {os.getcwd()}/{pdbf.replace('.pdb', '.fasta')} -db nr -out {os.getcwd()}/{pdbf.replace('.pdb', '.out')} -num_iterations 3 -evalue 0.001 -outfmt 11 -out_pssm {os.getcwd()}/{asnf} -save_pssm_after_last_round -num_threads 5")

## Predict

Edited utils.py:
- to adjust the fpocket callable to '../fpocket'
- pocket filenames to be 1-indexed instead of 0-indexed `in_file = open(self.path + self.pdbid + '_out/pockets/pocket' + str(pockidx+1) + '_atm.pdb')`
- np.mean of neighboring PSSM etc... adjusted so that np.mean() uses axis=0 and does column-average
- function sequenceCode because in the .asn from local psiblast the sequence is hexadecimal-encoded and it needs to be decoded for the rest to work

Edited AllosESmain.py to adjust the relative location of `models.m` 

In [7]:
for pdb, feats in extras_featuresd.items():
    # if pdb == "7sns": continue
    pdb = pdb.upper()
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    if os.path.isfile(f"{pdb}/{pdb}.asn") and not os.path.isfile(f"{pdb}/{pdb}_{chain}_result.csv"):
        os.system(f"cd {pdb} && python ../AllosES/AllosES/AllosESmain.py --PDBID {pdb} --CHAIN {chain}")

# Process

In [8]:
import pandas as pd

In [9]:
results = {}

for pdb, feats in extras_featuresd.items():
    # if pdb == "7sns": continue
    pdb = pdb.upper()
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    csv = f"{pdb}/{pdb}_{chain}_result.csv"
    if os.path.isfile(csv):
        results[pdb.lower()] = {
            f"pocket{pocket.iloc[0]+1}": {
                "pro_ave": pocket["pro_ave"],
                "residues": pd.DataFrame(
                    (
                        res.split(":") 
                        for res in pocket["residues"].split(",")
                    ),
                    columns=["auth_seq_id", "auth_asym_id"]
                )
            }
            for i, pocket in pd.read_csv(csv).iterrows()
        }

len(results), results

(24,
 {'8sgj': {'pocket1': {'pro_ave': 0.4566435782802614,
    'residues':    auth_seq_id auth_asym_id
    0          837            A
    1          172            A
    2          833            A
    3          830            A
    4          829            A
    5          836            A
    6          840            A
    7          210            A
    8           99            A
    9          103            A
    10         214            A
    11         244            A
    12          97            A
    13         825            A
    14         827            A
    15         826            A
    16         168            A
    17         175            A
    18         171            A
    19         102            A
    20         165            A
    21         209            A
    22         213            A
    23          98            A
    24         222            A
    25         224            A},
   'pocket54': {'pro_ave': 0.2878497901019058,
    'residues': 

In [10]:
resultsf = "alloses_results.pkl"

with open(resultsf, "wb") as f:
    pickle.dump(results, f)